In [0]:
USE CATALOG logistics_prod;

-- =========================================================================
-- 1. CHANGE DATA FEED (CDF): Auditoría nativa fila por fila en Delta Lake
-- =========================================================================
ALTER TABLE logistics_prod.silver.logistics_clean
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Consultamos el historial de cambios de la tabla usando table_changes()
SELECT
    asset_id,
    shipment_status,
    temperature_c,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes('logistics_prod.silver.logistics_clean', 0)
LIMIT 15;

-- =========================================================================
-- 2. ROW-LEVEL SECURITY (RLS): Filtro dinámico por Centro Logístico (Hub)
-- =========================================================================
-- Creamos una función de filtro de filas:
-- Si es miembro de 'global_logistics_managers' o es el dueño del catálogo, ve todos los Hubs;
-- si no, solo ve los envíos de 'Madrid Hub'.
CREATE OR REPLACE FUNCTION logistics_prod.gold.filter_by_assigned_hub(hub_name STRING)
RETURN
    is_account_group_member('global_logistics_managers')
    OR CURRENT_USER() = 'lcampo25@gmail.com'
    OR hub_name = 'Madrid Hub';

-- Aplicamos el filtro de fila directamente sobre la tabla de predicciones ML en Gold
ALTER TABLE logistics_prod.gold.ml_delay_predictions
SET ROW FILTER logistics_prod.gold.filter_by_assigned_hub ON (assigned_hub);

-- Verificamos que la consulta aplica la política de seguridad transparentemente
SELECT assigned_hub, COUNT(*) AS total_predictions, ROUND(AVG(delay_probability_pct), 2) AS avg_risk_pct
FROM logistics_prod.gold.ml_delay_predictions
GROUP BY assigned_hub;
